[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/b_02_vmap_batching.ipynb)

# 🟢 Easy: Pairwise Distances with vmap

*JAX Fundamentals*
Given `X` of shape `(N, D)` and `Y` of shape `(M, D)`, compute the matrix of
**squared Euclidean distances** of shape `(N, M)`:

$$D_{ij} = \|x_i - y_j\|_2^2 = \sum_{d} (X_{id} - Y_{jd})^2$$

The point of this problem is *not* the math — it is learning to write the
function for a **single example** and let `jax.vmap` add the batch dimensions.
This "write one, vmap the rest" habit is what interviewers are looking for.

### Rules
- Write a single-pair helper, then compose **two** `vmap`s around it
- No Python `for` loops over N or M
- Do **not** use the expand-dims broadcasting trick (`X[:, None] - Y[None]`);
  the exercise is specifically about `vmap` and `in_axes`
- Do not use `jnp.linalg.norm` or `scipy` distance helpers

### Example
```
X shape (3, 2), Y shape (5, 2)  ->  output shape (3, 5)
```

### Why it matters
Not for speed — under `jit` XLA fuses both spellings into the same kernel, and
neither actually materializes the `(N, M, D)` intermediate. (You can check:
`jax.jit(f).lower(X, Y).compile().memory_analysis()` reports the same temp
buffers for both.)

The win is that you never hand-manage axes. The `vmap` version is written for
one pair of vectors and stays readable when the batching gets harder — add a
head axis, a device axis, a per-example gradient — where the expand-dims
version turns into a pile of `None` indices you have to re-derive every time.
And it composes: `vmap(grad(f))` gives per-example gradients for free.

Being fluent with `in_axes=(None, 0)` vs `(0, None)` is a very common JAX
screening question.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def pairwise_sq_dist(X, Y):
    """Squared Euclidean distances between every row of X and every row of Y.

    Args:
        X: (N, D) array
        Y: (M, D) array

    Returns:
        (N, M) array where out[i, j] = ||X[i] - Y[j]||^2
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax.numpy as jnp

X = jnp.array([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0]])
Y = jnp.array([[0.0, 0.0], [1.0, 1.0]])

out = pairwise_sq_dist(X, Y)
print("X:", X.shape, " Y:", Y.shape)
print("out shape:", out.shape, "(expected (3, 2))")
print(out)
# Row 0 is distance from origin to each Y -> [0., 2.]

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution, status

check("vmap_batching")

# hint("vmap_batching")      # stuck? nudge without the answer
# solution("vmap_batching")  # spoiler: the reference implementation
# status()                   # your dashboard across all problems